# Capítulo 21 · Laboratório 21: SMC e manipulação
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**. Na primeira execução, o notebook baixa os dados do Yahoo Finance na sua máquina e guarda uma cópia local (`dados_cap21.pkl`), que as execuções seguintes reaproveitam. O provedor corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. O gráfico de 60 minutos do livro vai de 07/11/2024 a 30/09/2026; o Yahoo só guarda 730 dias dessas barras, e o notebook avisa quando a janela muda. O laboratório testa as ideias do SMC (varredura de liquidez, CHoCH, FVG e *order block*) contra o acaso, confere a frase "o Ibovespa cai 10% todo ano" e mede os rompimentos falsos do box "Manipulação ou leilão?": frequência, liquidez, volume, profundidade das varreduras em ATR e o viés de olhar o futuro. As regras são públicas e simplificadas; outros autores de SMC usam definições diferentes.

In [ ]:
!pip install -q yfinance

In [ ]:
import logging
import os
import numpy as np
import pandas as pd
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
FIM = "2026-10-01"                      # dados até 30/09/2026
CACHE = "dados_cap21.pkl"


def confere(baixados):
    """O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local."""
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f"Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou"
                               " limitando pedidos. Espere alguns minutos e rode a célula de novo.")


pc = lambda x, k=1: "" if pd.isna(x) else f"{x:.{k}%}".replace(".", ",")
ps = lambda x, k=2: "" if pd.isna(x) else ("+" if x >= 0 else "") + pc(x, k)
nn = lambda x: f"{x:,}".replace(",", ".")
B3 = ["PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "WEGE3", "ABEV3", "B3SA3", "RENT3", "SUZB3",
      "GGBR4", "PRIO3", "RADL3", "EQTL3", "VIVT3", "LREN3", "EMBJ3", "CSAN3", "RAIL3", "CMIG4"]
EUA = ["AAPL", "MSFT", "JPM", "JNJ", "PG", "KO", "XOM", "CVX", "WMT", "HD", "MRK", "PFE", "CSCO", "INTC",
       "IBM", "MMM", "CAT", "DIS", "MCD", "BA"]
TICKERS = [t + ".SA" for t in B3] + EUA
MERC = {t: ("B3" if t.endswith(".SA") else "EUA") for t in TICKERS}
INTRA = ["SPY", "QQQ", "BOVA11.SA", "PETR4.SA", "VALE3.SA"]
K = 3          # pivô (fractal): mínima menor que as 3 barras de cada lado; só vale depois de confirmado

## Parte 1: dados

In [ ]:
if os.path.exists(CACHE):
    D, IDX, DX, I = pd.read_pickle(CACHE)
else:
    D = yf.download(TICKERS, start="2005-01-01", end=FIM, auto_adjust=True, progress=False, group_by="ticker")
    IDX = yf.download(["^BVSP", "^GSPC"], start="1997-01-01", end=FIM, auto_adjust=True, progress=False,
                      group_by="ticker")
    # exemplos das figuras: preços da época (sem ajuste por proventos), como na plataforma
    DX = yf.download([t + ".SA" for t in B3], start="2022-01-01", end=FIM, auto_adjust=False, progress=False,
                     group_by="ticker")
    # o Yahoo só guarda ~730 dias de barras de 60 minutos, contados a partir de hoje
    hoje = pd.Timestamp.today().normalize()
    ini60 = max((hoje - pd.Timedelta(days=720)).strftime("%Y-%m-%d"), "2024-11-07")   # o livro usa 07/11/2024 a 30/09/2026
    if ini60 < FIM:
        I = yf.download(INTRA, start=ini60, end=FIM, interval="60m", auto_adjust=True, progress=False, group_by="ticker")
    else:                               # FIM já saiu da janela: usa os últimos 720 dias até hoje
        I = yf.download(INTRA, start=ini60, interval="60m", auto_adjust=True, progress=False, group_by="ticker")
    confere({"as 40 ações": D, "o Ibovespa e o S&P 500": IDX, "as ações dos exemplos": DX, "as barras de 60 minutos": I})
    if ini60 > "2024-11-07" or ini60 >= FIM:
        print(f"Aviso: a janela do livro é de 07/11/2024 a 30/09/2026; o Yahoo só guarda 730 dias de barras de 60 minutos;"
              f" a Parte 2 usa de {I.index[0]:%d/%m/%Y} a {I.index[-1]:%d/%m/%Y}, e os números diferem dos do livro.")
    pd.to_pickle((D, IDX, DX, I), CACHE)
print(f"Parte 1. {len(TICKERS)} ações (20 da B3 e 20 dos EUA) de 2005 a {D.dropna(how='all').index[-1]:%d/%m/%Y};"
      f" Ibovespa e S&P 500 desde 1997; {len(INTRA)} ativos no gráfico de 60 minutos"
      f" ({I.index[0]:%d/%m/%Y} a {I.index[-1]:%d/%m/%Y}).")


def limpa(d):
    d = d.drop(columns=["Adj Close"], errors="ignore").dropna()
    return d[(d["Close"] > 0) & (d["High"] >= d["Low"])]


def pivos(h, l):
    """Último topo e último fundo de swing já confirmados em cada barra (sem olhar o futuro)."""
    janela = 2 * K + 1
    topo = h == h.rolling(janela, center=True).max()
    fundo = l == l.rolling(janela, center=True).min()
    ult_topo = h.where(topo).shift(K).ffill()      # o pivô da barra i só é conhecido na barra i + K
    ult_fundo = l.where(fundo).shift(K).ffill()
    return ult_topo, ult_fundo


def espaca(idx, n, pos):
    """Um sinal a cada n barras por ativo."""
    fica, ultimo = [], -10**9
    for x in idx:
        if pos[x] - ultimo >= n:
            fica.append(x)
            ultimo = pos[x]
    return fica


def varreduras(d, H=(5, 10, 20)):
    o, h, l, c = d["Open"], d["High"], d["Low"], d["Close"]
    ut, uf = pivos(h, l)
    pos = pd.Series(np.arange(len(d)), index=d.index)
    fut = {k: c.shift(-k) / c - 1 for k in H}
    sinais = {"varredura de fundo (fechou de volta)": (l < uf) & (c > uf),
              "quebra do fundo (fechou abaixo)": c < uf,
              "varredura de topo (fechou de volta)": (h > ut) & (c < ut),
              "quebra do topo (fechou acima)": c > ut}
    linhas = []
    for nome, s in sinais.items():
        for x in espaca(s[s.fillna(False)].index, 10, pos):
            linhas.append({"sinal": nome, **{f"r{k}": fut[k].get(x) for k in H}})
            if nome.startswith("varredura de fundo"):           # varredura + CHoCH: em até 10 barras,
                p, alvo, piso = pos[x], ut.loc[x], l.loc[x]       # fecha acima do último topo, sem perder a mínima
                for q in range(p + 1, min(p + 11, len(d))):
                    if l.iloc[q] < piso:
                        break
                    if c.iloc[q] > alvo:
                        y = d.index[q]
                        linhas.append({"sinal": "varredura de fundo + CHoCH (entrada na quebra)",
                                       **{f"r{k}": fut[k].get(y) for k in H}})
                        break
    base = pd.DataFrame({f"r{k}": fut[k] for k in H}).dropna()
    base["sinal"] = "qualquer barra"
    return pd.concat([pd.DataFrame(linhas), base], ignore_index=True)


sorteio = np.random.default_rng(0)


def fvgs(d):
    """FVG de alta: mínima da barra t acima da máxima da barra t-2. Testa o retorno ao vazio e a reação."""
    h, l, c = d["High"], d["Low"], d["Close"]
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    atr = tr.rolling(14).mean()
    gap = (l > h.shift(2)) & ((l - h.shift(2)) >= 0.2 * atr)      # vazio de pelo menos 0,2 ATR
    linhas, n = [], len(d)
    cv, lv, hv, gapv = c.to_numpy(), l.to_numpy(), h.to_numpy(), gap.fillna(False).to_numpy()
    for i in np.flatnonzero(gapv):
        if i + 41 > n:
            break
        topo_gap, fundo_gap = lv[i], hv[i - 2]
        dist = (cv[i] - topo_gap) / cv[i]                           # distância do fechamento até o vazio
        futuro_l = lv[i + 1:i + 21]
        toca = np.flatnonzero(futuro_l <= topo_gap)
        # nível de controle: mesma distância percentual abaixo do fechamento de uma barra sorteada do mesmo ativo
        j = int(sorteio.integers(3, n - 31))
        nivel = cv[j] * (1 - dist)
        tc = np.flatnonzero(lv[j + 1:j + 21] <= nivel)
        toca_ctrl = len(tc) > 0
        reacao_ctrl = cv[j + 1 + tc[0] + 10] / nivel - 1 if len(tc) else np.nan
        reacao = np.nan
        if len(toca):
            t = i + 1 + toca[0]
            reacao = cv[t + 10] / max(topo_gap, lv[t]) - 1 if t + 10 < n else np.nan
        linhas.append({"toca": len(toca) > 0, "preenche": (futuro_l <= fundo_gap).any(),
                       "toca_ctrl": toca_ctrl, "reacao10": reacao, "reacao_ctrl": reacao_ctrl})
    return pd.DataFrame(linhas)


def order_blocks(d):
    """Rompimento forte: fecha acima da máxima de 20 barras com barra maior que 1,5 ATR.
    Order block: a última barra de baixa das 5 anteriores. Antiga resistência: a máxima de 20 barras rompida.
    As duas zonas têm a mesma largura (a da barra do order block), para a comparação ser justa.
    Para cada zona, o primeiro reteste em até 30 barras e o que acontece 10 barras depois."""
    o, h, l, c = d["Open"], d["High"], d["Low"], d["Close"]
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    atr = tr.rolling(14).mean()
    max20 = h.rolling(20).max().shift(1)
    forte = (c > max20) & ((h - l) > 1.5 * atr)
    linhas, n = [], len(d)
    pos = pd.Series(np.arange(n), index=d.index)
    for x in espaca(forte[forte.fillna(False)].index, 20, pos):
        b = pos[x]
        if b < 6 or b + 41 > n:
            continue
        baixa = [k for k in range(b - 5, b) if c.iloc[k] < o.iloc[k]]
        if not baixa:
            continue
        k = baixa[-1]
        largura = h.iloc[k] - l.iloc[k]                                  # as duas zonas com a mesma largura
        zonas = {"order block (última barra de baixa)": (h.iloc[k], l.iloc[k]),
                 "antiga resistência (Cap. 4)": (max20.iloc[b], max20.iloc[b] - largura)}
        for nome, (topo, fundo) in zonas.items():
            if topo >= c.iloc[b]:
                continue
            for t in range(b + 1, b + 31):
                if l.iloc[t] <= topo:                                   # reteste: tocou o topo da zona
                    segurou = c.iloc[t + 1:t + 11].min() >= fundo        # 10 barras sem fechar abaixo da zona
                    linhas.append({"zona": nome, "segurou": segurou,
                                   "r10": c.iloc[t + 10] / topo - 1})
                    break
            else:
                linhas.append({"zona": nome, "segurou": np.nan, "r10": np.nan, "sem_reteste": True})
    return pd.DataFrame(linhas)

## Parte 2: varreduras de liquidez

In [ ]:
V, F, O = [], [], []
for t in TICKERS:
    d = limpa(D[t])
    if len(d) < 500:
        continue
    V.append(varreduras(d)); F.append(fvgs(d)); O.append(order_blocks(d))
V, F, O = pd.concat(V), pd.concat(F), pd.concat(O)

ORDEM = ["varredura de fundo (fechou de volta)", "varredura de fundo + CHoCH (entrada na quebra)",
         "quebra do fundo (fechou abaixo)", "varredura de topo (fechou de volta)", "quebra do topo (fechou acima)",
         "qualquer barra"]


def tabela(V, rotulo):
    print(rotulo)
    print("   sinal                                              casos |  5 barras       | 10 barras       | 20 barras")
    res = {}
    for nome in ORDEM:
        x = V[V["sinal"] == nome].dropna()
        res[nome] = x
        print(f"   {nome:<50} {nn(len(x)):>7} | " + " | ".join(
            f"{ps(x[f'r{k}'].mean()):>6} ({pc((x[f'r{k}'] > 0).mean(), 0)})" for k in (5, 10, 20)))
    return res


TV = tabela(V, "\nParte 2. Varreduras de liquidez no diário (40 ações, desde 2005; pivô de 3 barras de cada lado)")
VI = pd.concat([varreduras(limpa(I[t])) for t in INTRA])
TVI = tabela(VI, "\n   Mesmo teste no gráfico de 60 minutos (SPY, QQQ, BOVA11, PETR4, VALE3; retornos em barras de 60 min)")

## Parte 3: FVG

In [ ]:
print(f"\nParte 3. Fair Value Gaps de alta no diário: {nn(len(F))} casos")
print(f"   voltou a tocar o vazio em até 20 barras: {pc(F['toca'].mean(), 0)} | preencheu o vazio inteiro:"
      f" {pc(F['preenche'].mean(), 0)}")
print(f"   nível de controle (mesma distância, barra sorteada): tocado em {pc(F['toca_ctrl'].mean(), 0)}")
r, rc = F["reacao10"].dropna(), F["reacao_ctrl"].dropna()
print(f"   depois de tocar o FVG, 10 barras: média {ps(r.mean())}, positivo em {pc((r > 0).mean(), 0)} ({nn(len(r))} casos)")
print(f"   depois de tocar o nível de controle, 10 barras: média {ps(rc.mean())}, positivo em {pc((rc > 0).mean(), 0)}"
      f" ({nn(len(rc))} casos)")

## Parte 4: order block x antiga resistência

In [ ]:
print("\nParte 4. Order block x antiga resistência, depois de rompimento forte (diário)")
OB = {}
for nome, x in O.groupby("zona"):
    com = x.dropna(subset=["r10"])
    OB[nome] = (len(x), len(com) / len(x), com["segurou"].astype(float).mean(), com["r10"].mean())
    print(f"   {nome:<38} {nn(len(x)):>6} rompimentos | reteste em 30 barras: {pc(len(com) / len(x), 0)}"
          f" | segurou 10 barras: {pc(com['segurou'].astype(float).mean(), 0)} | 10 barras depois: {ps(com['r10'].mean())}")

## Parte 5: "o Ibovespa cai 10% todo ano"

In [ ]:
ib = IDX["^BVSP"]["Close"].dropna()
sp = IDX["^GSPC"]["Close"].dropna()
QIB = pd.Series({a: (x / x.cummax() - 1).min() for a, x in ib.groupby(ib.index.year)})
QSP = pd.Series({a: (x / x.cummax() - 1).min() for a, x in sp.groupby(sp.index.year)})
completos, qs = QIB.loc[:ib.index[-1].year - 1], QSP.loc[:sp.index[-1].year - 1]
print("\nParte 5. \"Desde 1997 o Ibovespa cai 10% todo ano\": maior queda de um topo a um fundo dentro de cada ano")
print(f"   anos completos com queda de 10% ou mais: {(completos <= -0.10).sum()} de {len(completos)}")
print("   anos que ficaram abaixo de 10%: " + ", ".join(f"{a} ({pc(v, 0)})" for a, v in completos[completos > -0.10].items()))
print(f"   {ib.index[-1].year} até {ib.index[-1]:%d/%m}: maior queda de {pc(QIB.iloc[-1], 0)}")
print(f"   S&P 500 no mesmo período: {(qs <= -0.10).sum()} de {len(qs)} anos com queda de 10% ou mais"
      f" | queda mediana dentro do ano: Ibovespa {pc(completos.median(), 0)}, S&P 500 {pc(qs.median(), 0)}")

In [ ]:
import matplotlib.pyplot as plt
ax = QIB.plot(kind="bar", figsize=(10, 3.5), color="black", title="Maior queda dentro de cada ano: Ibovespa")
ax.axhline(-0.10, color="gray", ls="--")
plt.show()

## Parte 6: rompimentos falsos (box "Manipulação ou leilão?")

In [ ]:
N, CORTE = 20, pd.Timestamp("2016-01-01")


def limpa_v(d):
    d = limpa(d)
    ok = (d[["Open", "High", "Low", "Close"]] > 0).all(axis=1) & (d["High"] > d["Low"]) & (d["Volume"] > 0)
    return d[ok].copy()


def atr(d, n=20):
    h, l, c = d["High"], d["Low"], d["Close"]
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1 / n, adjust=False).mean()


BASE = {t: limpa_v(D[t]) for t in TICKERS}

# liquidez relativa: volume financeiro médio de 60 dias, em tercis dentro de cada mercado, a cada dia
volfin = {t: (BASE[t]["Close"] * BASE[t]["Volume"]).rolling(60).mean() for t in TICKERS}
TERCIL = {}
for m in ("B3", "EUA"):
    vf = pd.DataFrame({t: volfin[t] for t in TICKERS if MERC[t] == m})
    rk = vf.rank(axis=1, pct=True)
    for t in vf.columns:
        TERCIL[t] = pd.cut(rk[t], [0, 1 / 3, 2 / 3, 1.0], labels=["menos líquidas", "meio", "mais líquidas"])

ev = []
for t in TICKERS:
    d = BASE[t]
    o, h, l, c, v = d["Open"], d["High"], d["Low"], d["Close"], d["Volume"]
    H, L = h.rolling(N).max().shift(1), l.rolling(N).min().shift(1)
    a = atr(d).shift(1)
    rvol = v / v.rolling(N).mean().shift(1)
    fut = {k: c.shift(-k) / c - 1 for k in (5, 10, 20)}                   # a partir do fechamento do dia do rompimento
    fut3 = {k: c.shift(-3 - k) / c.shift(-3) - 1 for k in (5, 10, 20)}    # a partir do 3º pregão depois (tipo já conhecido)
    base = {k: fut[k].mean() for k in fut}
    # fechou de volta para dentro do canal em até 3 pregões depois de ter fechado fora
    volta_cima = pd.concat([(c.shift(-k) <= H) for k in (1, 2, 3)], axis=1).any(axis=1)
    volta_baixo = pd.concat([(c.shift(-k) >= L) for k in (1, 2, 3)], axis=1).any(axis=1)
    for lado, toque, fora, nivel, volta in (("alta", h > H, c > H, H, volta_cima), ("baixa", l < L, c < L, L, volta_baixo)):
        sel = toque & H.notna() & a.notna()
        for dt in d.index[sel]:
            if fora[dt]:
                tipo = "falso em 3 dias" if volta[dt] else "verdadeiro"
            else:
                tipo = "falso no dia"
            prof = ((h[dt] - nivel[dt]) if lado == "alta" else (nivel[dt] - l[dt])) / a[dt]
            ev.append({"ação": t, "mercado": MERC[t], "data": dt, "lado": lado, "tipo": tipo, "rvol": rvol[dt],
                       "prof_atr": prof, "liquidez": TERCIL[t].get(dt),
                       **{f"r{k}": fut[k][dt] - base[k] for k in fut},
                       **{f"d3r{k}": fut3[k][dt] - base[k] for k in fut3},
                       "volta10": (c.shift(-10)[dt] > nivel[dt]) if lado == "baixa" else (c.shift(-10)[dt] < nivel[dt])})
EV = pd.DataFrame(ev).dropna(subset=["r20", "d3r20"])
EV["período"] = np.where(EV["data"] < CORTE, "2005-2015", "2016-2026")

# frequência dos rompimentos falsos
print(f"\nParte 6. Rompimentos do canal de {N} dias (máxima ou mínima), 40 ações, 2005 a 2026: {nn(len(EV))} eventos")
t1 = EV.groupby(["lado", "tipo"]).size().unstack()
t1 = t1[["falso no dia", "falso em 3 dias", "verdadeiro"]]
for lado, lin in t1.iterrows():
    tot = lin.sum()
    print(f"   {lado:5s}: falso no dia {pc(lin['falso no dia'] / tot, 0)} | fechou fora e voltou em até 3 pregões"
          f" {pc(lin['falso em 3 dias'] / tot, 0)} | verdadeiro {pc(lin['verdadeiro'] / tot, 0)} (n = {nn(int(tot))})")

# o que acontece depois, e o viés de olhar o futuro
print("\n   Retorno depois do rompimento, contado a partir do 3º pregão (quando já se sabe se foi falso),")
print("   menos o retorno de um dia qualquer da mesma ação (ponto de vista de quem está comprado)")
t2 = EV.groupby(["lado", "tipo", "período"])[["d3r5", "d3r10", "d3r20"]].mean().unstack("período")
for (lado, tipo), lin in t2.iterrows():
    print(f"   {lado:5s} {tipo:16s}: 20 pregões {pc(lin[('d3r20', '2005-2015')], 2):>7s} (2005-2015) |"
          f" {pc(lin[('d3r20', '2016-2026')], 2):>7s} (2016-2026) | 5 pregões {pc(lin[('d3r5', '2016-2026')], 2):>7s} (2016-2026)")

print("\n   O viés de olhar o futuro: a mesma conta a partir do dia do rompimento, classificando com os 3 pregões seguintes")
t2b = EV.groupby(["lado", "tipo"])["r20"].mean()
for (lado, tipo), v in t2b.items():
    print(f"   {lado:5s} {tipo:16s}: 20 pregões {ps(v)}")
print("   (parece vantagem, mas a classificação usa os 3 pregões seguintes, que ainda não eram conhecidos no dia)")

## Parte 7: liquidez

In [ ]:
print("\nParte 7. Parcela de rompimentos falsos (no dia ou em 3 pregões) por liquidez relativa")
EV["falso"] = EV["tipo"] != "verdadeiro"
t3 = EV.dropna(subset=["liquidez"]).groupby(["mercado", "liquidez"], observed=True)["falso"].agg(["mean", "size"])
for (m, q), lin in t3.iterrows():
    print(f"   {m:3s} {q:15s}: {pc(lin['mean'], 1)} falsos (n = {nn(int(lin['size']))})")

## Parte 8: volume

In [ ]:
print("\nParte 8. Volume do dia do rompimento (volume relativo à média de 20 dias) e resultado")
fora = EV[EV["tipo"] != "falso no dia"].copy()          # fechou fora do canal no dia
fora["faixa_vol"] = pd.cut(fora["rvol"], [0, 1, 1.5, 2, 1e9], labels=["< 1x", "1x a 1,5x", "1,5x a 2x", "> 2x"])
t4 = fora.groupby(["lado", "faixa_vol"], observed=True).agg(n=("r20", "size"), falso=("falso", "mean"), r20=("r20", "mean"))
for (lado, fx), lin in t4.iterrows():
    sinal = 1 if lado == "alta" else -1
    print(f"   {lado:5s} volume {fx:10s}: n = {nn(int(lin['n'])):>6s} | voltou em 3 pregões {pc(lin['falso'], 0)} |"
          f" 20 pregões no sentido do rompimento {pc(sinal * lin['r20'], 2)}")
med = EV.groupby("tipo")["rvol"].median()
print(f"   volume relativo mediano: falso no dia {med['falso no dia']:.2f}x | falso em 3 dias {med['falso em 3 dias']:.2f}x |"
      f" verdadeiro {med['verdadeiro']:.2f}x".replace(".", ","))

## Parte 9: stops óbvios

In [ ]:
print("\nParte 9. Stop colado na mínima de 20 dias: quanto o preço passa dela (em ATR) antes de voltar")
fu = EV[EV["lado"] == "baixa"].copy()
fu["voltou"] = fu["tipo"] == "falso no dia"
print(f"   mínimas furadas: {nn(len(fu))}; fecharam de volta acima no mesmo dia: {pc(fu['voltou'].mean(), 0)}")
q = fu[fu["voltou"]]["prof_atr"].quantile([0.5, 0.75, 0.9])
print(f"   profundidade das varreduras (fecharam de volta): mediana {q[0.5]:.2f} ATR | 75% até {q[0.75]:.2f} | 90% até {q[0.9]:.2f}"
      .replace(".", ","))
for k in (0.25, 0.5, 1.0):
    salva = (fu[fu["voltou"]]["prof_atr"] < k).mean()
    custo = fu[~fu["voltou"]]["prof_atr"].ge(k).mean()
    kk = f"{k:.2f}".replace(".", ",")
    print(f"   stop a {kk} ATR abaixo da mínima: escapa de {pc(salva, 0)} das varreduras;"
          f" no dia das quebras de verdade, o preço já passou dessa folga em {pc(custo, 0)} delas (custo: {kk} ATR a mais de perda)")
print(f"   depois da varredura (fechou de volta), o preço estava acima da mínima 10 pregões depois em {pc(fu[fu['voltou']]['volta10'].mean(), 0)}"
      f" dos casos; depois da quebra de verdade, em {pc(fu[~fu['voltou']]['volta10'].mean(), 0)}")

## Parte 10: scanner de molas, upthrusts e varreduras com CHoCH

In [ ]:
def scanner(t, ini="2023-01-01"):
    d = BASE[t]
    a = atr(d)
    topo, fundo = d["High"].rolling(40).max().shift(1), d["Low"].rolling(40).min().shift(1)
    largura = (topo - fundo) / a.shift(1)
    lateral = largura.between(4, 9)
    meio = (topo + fundo) / 2
    out = []
    for dt in d.index[(d.index >= ini) & lateral.fillna(False)]:
        i = d.index.get_loc(dt)
        if i + 20 >= len(d):
            continue
        lo, hi, cl = d["Low"].iloc[i], d["High"].iloc[i], d["Close"].iloc[i]
        fut = d["Close"].iloc[i + 1:i + 21]
        sem_salto = (d["Open"].iloc[i + 1:i + 21] / d["Close"].iloc[i:i + 20].to_numpy() - 1).abs().max() < 0.04
        if lo < fundo.iloc[i] and cl > fundo.iloc[i]:
            prof = (fundo.iloc[i] - lo) / a.iloc[i - 1]
            ok = fut.iloc[-1] > meio.iloc[i]
            limpo = ok and not (fut.iloc[:10] < fundo.iloc[i]).any() and sem_salto
            falhou = (fut.iloc[:10] < fundo.iloc[i]).any() and fut.iloc[-1] < fundo.iloc[i]
            out.append(("mola", t, dt, prof, ok, limpo, falhou and sem_salto, falhou, fut.iloc[-1] / cl - 1))
        if hi > topo.iloc[i] and cl < topo.iloc[i]:
            prof = (hi - topo.iloc[i]) / a.iloc[i - 1]
            ok = fut.iloc[-1] < meio.iloc[i]
            limpo = ok and not (fut.iloc[:10] > topo.iloc[i]).any() and sem_salto
            falhou = (fut.iloc[:10] > topo.iloc[i]).any() and fut.iloc[-1] > topo.iloc[i]
            out.append(("upthrust", t, dt, prof, ok, limpo, falhou and sem_salto, falhou, fut.iloc[-1] / cl - 1))
    return out


SC = pd.DataFrame([x for t in TICKERS if MERC[t] == "B3" for x in scanner(t)],
                  columns=["tipo", "ação", "data", "prof_atr", "deu certo", "limpo", "falhou limpo", "falhou", "r20"])
SC = SC[SC["prof_atr"].between(0.1, 1.2)]
SC["mês"] = SC["data"].dt.to_period("M")
SC = SC.drop_duplicates(["tipo", "ação", "mês"]).drop(columns="mês")
print(f"\nParte 10. Scanner na B3 (2023 a 2026, faixa lateral de 40 dias entre 4 e 9 ATR): {len(SC)} candidatos")
for tipo in ("mola", "upthrust"):
    g = SC[SC["tipo"] == tipo]
    print(f"   {tipo}: {len(g)} casos | deram certo (passaram do meio da faixa em 20 pregões) {pc(g['deu certo'].mean(), 0)}"
          f" | falharam (romperam de verdade) {pc(g['falhou'].mean(), 0)}")
ESCOLHA = {}
# para as figuras: casos "limpos" (sem voltar a romper em 10 pregões e sem salto de abertura maior que 4%,
# para não confundir com notícia ou balanço)
for rot, filtro in (("mola que deu certo", (SC["tipo"] == "mola") & SC["limpo"]),
                    ("upthrust que deu certo", (SC["tipo"] == "upthrust") & SC["limpo"]),
                    ("mola que falhou", (SC["tipo"] == "mola") & SC["falhou limpo"])):
    g = SC[filtro].copy()
    g["força"] = g["r20"].abs()
    g = g.sort_values("força", ascending=False).drop_duplicates("ação")
    print(f"   candidatos: {rot}")
    for _, x in g.head(5).iterrows():
        pa = f"{x['prof_atr']:.2f}".replace(".", ",")
        print(f"      {x['ação'][:-3]:6s} {x['data']:%d/%m/%Y}  passou {pa} ATR  20 pregões depois {pc(x['r20'])}")
    if len(g):
        ESCOLHA[rot] = g.iloc[0]

# varredura de fundo seguida de CHoCH na B3 (2023 a 2026): estatística com preços ajustados (como o resto do teste);
# os exemplos das figuras saem da mesma conta com preços da época (DX), como na plataforma
def sequencias(fonte):
    out = []
    for t in [t + ".SA" for t in B3]:
        d = limpa(fonte[t])
        h, l, c = d["High"], d["Low"], d["Close"]
        ut, uf = pivos(h, l)
        pos = pd.Series(np.arange(len(d)), index=d.index)
        sv = (l < uf) & (c > uf) & (d.index >= "2023-01-01")
        for x in espaca(sv[sv.fillna(False)].index, 10, pos):
            p, alvo, piso = pos[x], ut.loc[x], l.loc[x]
            for q in range(p + 1, min(p + 11, len(d))):
                if l.iloc[q] < piso:
                    break
                if c.iloc[q] > alvo:
                    if q + 20 < len(d):
                        out.append({"ação": t, "varredura": x, "choch": d.index[q], "fundo": uf.loc[x], "topo": alvo,
                                    "mínima": piso, "r20": c.iloc[q + 20] / c.iloc[q] - 1,
                                    "perdeu a mínima": (l.iloc[q + 1:q + 21] < piso).any()})
                    break
    return pd.DataFrame(out)


EXA, EXV = sequencias(D), sequencias(DX)
print(f"\n   Varredura de fundo + CHoCH na B3 (2023 a 2026, preços ajustados): {len(EXA)} casos | 20 pregões depois da"
      f" entrada: média {ps(EXA['r20'].mean())}, positivos {pc((EXA['r20'] > 0).mean(), 0)} | perderam a mínima da"
      f" varredura {pc(EXA['perdeu a mínima'].mean(), 0)}")
for rot, g in (("deu certo", EXV[~EXV["perdeu a mínima"]].sort_values("r20", ascending=False)),
               ("falhou", EXV[EXV["perdeu a mínima"]].sort_values("r20"))):
    print(f"   candidatos para as figuras (preços da época): varredura + CHoCH que {rot}")
    for _, x in g.drop_duplicates("ação").head(4).iterrows():
        print(f"      {x['ação'][:-3]:6s} varredura {x['varredura']:%d/%m/%Y}  CHoCH {x['choch']:%d/%m/%Y}"
              f"  20 pregões depois {pc(x['r20'])}")

In [ ]:
import matplotlib.pyplot as plt
d = limpa(DX["LREN3.SA"]).loc["2025-02-01":"2025-06-10"]
ax = d["Close"].plot(figsize=(9, 3.5), color="black", title="LREN3: varredura de fundo e CHoCH (abril de 2025)")
for _, x in EXV[EXV["ação"] == "LREN3.SA"].iterrows():
    if d.index[0] <= x["varredura"] <= d.index[-1]:
        ax.axhline(x["fundo"], color="gray", ls="--", lw=0.8)
        ax.axhline(x["topo"], color="gray", lw=0.8)
        ax.annotate("varredura", (x["varredura"], x["mínima"]), fontsize=8, ha="center")
        ax.annotate("CHoCH", (x["choch"], x["topo"]), fontsize=8, ha="center")
plt.show()

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, a função `pivos` da Parte 1 e o uso dela dentro de `varreduras`. Mostre por que `rolling(janela, center=True)` sozinho olharia o futuro e como o `shift(K)` corrige isso: o pivô da barra i só é conhecido na barra i + K." Confira com a Parte 2, que imprime a Tabela 21.1 (varredura de fundo: +0,90% em 20 barras, em 5.658 casos; barra qualquer: +1,46%).
2. **Adaptar:** "Numa célula nova, depois da Parte 2, baixe [ATIVO] com `yf.download(['[TICKER]'], start='2005-01-01', end=FIM, auto_adjust=True, group_by='ticker')`, limpe a série com `limpa` e rode `tabela(varreduras(d), '[ATIVO]')`. Não mude `K = 3` nem as regras de `varreduras`." Antes de olhar o ativo novo, confira se a Parte 2 continua imprimindo os números do livro para as 40 ações.

Mais pedidos deste capítulo (entender e aplicar ao seu gráfico) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 2, troque o pivô de 3 para 5 barras (K = 5). A varredura com CHoCH melhora?
2. Na Parte 3, exija um FVG de pelo menos 0,5 ATR. A diferença para o nível qualquer aparece?
3. Na Parte 4, defina o order block como as duas últimas barras de baixa. O resultado muda?
4. Na Parte 8, separe B3 e EUA. O volume ajuda nos dois mercados?
5. Na Parte 9, teste folgas de 0,3 e 0,7 ATR e calcule o custo em reais numa ação que você acompanha.

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.